# RU-002A — Tunable MZI with 2×2 MMI couplers (SiEPIC OpenEBL target)

**Status: prepared simulation notebook; not yet executed.** This is an EIS adaptation scaffold inspired by FlexCompute's [Tunable MZI tutorial](https://www.flexcompute.com/tidy3d/community/notebooks/TunableMZI/). It is not a verified reproduction of the CUMEC chip or a foundry-qualified SiEPIC layout.

## Experiment question
Can a parameterized MZI made from two multimode-interference (MMI) regions, two arms, and a tunable phase section produce the expected phase-dependent transfer between two output modes in Tidy3D FDTD?

## Important scope limits
- The starting MMI dimensions below are **search seeds, not validated 50:50 couplers**. Sweep and validate them before interpreting the MZI result.
- This notebook uses a simplified homogeneous SOI-like optical model (Si core / SiO₂ cladding). It does **not** load or certify the SiEPIC OpenEBL PDK, and it does not use a verified PhotonForge PDK component. Confirm the installed PDK API and actual layer/material stack on the licensed host before calling this a PDK-based result.
- The phase sweep changes the phase-section core index as a numerical proxy. It is **not** a thermal/TiN heater model, and `delta_n` is not a measured effective-index-versus-power calibration.
- No Tidy3D cloud job is launched unless `RUN_REMOTE = True` is deliberately set. No simulation results are included in this committed notebook.
- An MZI transfer function alone does not establish MLA/KKA equivalence. That requires a separately pinned encoding, workload, and electronic baseline.


## 1. Imports and run authorization

Install the supported Tidy3D version and configure credentials on the authorized host using FlexCompute's official instructions. Do not put API keys in the notebook or repository.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import tidy3d as td
import tidy3d.web as web

RUN_REMOTE = False  # Set True only after reviewing geometry, cost, credentials, and PDK assumptions.
OUT_DIR = Path('ru002a_mmi_mzi_results')
OUT_DIR.mkdir(exist_ok=True)
print('Tidy3D version:', td.__version__)
print('Remote simulation enabled:', RUN_REMOTE)


## 2. Explicit initial geometry parameters

Propagation is along `x`; transverse in-plane direction is `y`; the silicon core is centered at `z=0`. The two MMI regions are represented initially as widened rectangular silicon sections. This deliberately simple geometry makes the design assumptions visible; it does not imply that the dimensions achieve a 50:50 split. Validate mesh convergence, port placement, and MMI length/width by a bounded sweep.

In [ ]:
# Length units: micrometres. Wavelength: 1.55 um.
lambda0 = 1.55
freq0 = td.C_0 / lambda0
wavelengths = np.linspace(1.53, 1.57, 5)
freqs = td.C_0 / wavelengths

# Optical stack: simplified lossless material model, not a PDK-certified stack.
n_si = 3.48
n_sio2 = 1.44
si = td.Medium(permittivity=n_si**2)
sio2 = td.Medium(permittivity=n_sio2**2)

# Geometry seed values. Treat MMI length/width/port pitch as sweep variables.
wg_width = 0.45
wg_thickness = 0.22
mmi_length = 18.0
mmi_width = 6.0
port_pitch = 2.0
lead_length = 5.0
arm_length = 20.0
phase_length = 10.0
phase_delta_n = 0.0  # Sweep as a numerical phase proxy; not heater power/temperature.

mmi_x1 = -0.5 * (arm_length + mmi_length)
mmi_x2 = +0.5 * (arm_length + mmi_length)
x_in = mmi_x1 - 0.5 * mmi_length - lead_length
x_out = mmi_x2 + 0.5 * mmi_length + lead_length
y_top = +port_pitch / 2
y_bottom = -port_pitch / 2

print({'lambda0_um': lambda0, 'mmi_length_um': mmi_length, 'mmi_width_um': mmi_width,
       'port_pitch_um': port_pitch, 'phase_length_um': phase_length,
       'phase_delta_n_proxy': phase_delta_n})


## 3. Build a two-MMI MZI

The access waveguides connect to the ends of the MMI blocks. For a production-quality device, replace abrupt interfaces with the verified PDK's component geometry or validated tapers. Here the model is intended for initial interference characterization only.

In [ ]:
def box(center, size, medium, name=None):
    geom = td.Box(center=center, size=size)
    return td.Structure(geometry=geom, medium=medium, name=name) if name else td.Structure(geometry=geom, medium=medium)

def make_mzi_sim(delta_n=0.0, mmi_length_um=mmi_length, mmi_width_um=mmi_width,
                 port_pitch_um=port_pitch, grid_steps_per_wvl=16):
    """Build an initial 3D FDTD model. MMI geometry is a seed, not pre-optimized."""
    x1 = -0.5 * (arm_length + mmi_length_um)
    x2 = +0.5 * (arm_length + mmi_length_um)
    y1, y2 = +port_pitch_um / 2, -port_pitch_um / 2
    wg_size_z = wg_thickness
    structures = []

    # Two rectangular multimode sections.
    structures.append(box((x1, 0, 0), (mmi_length_um, mmi_width_um, wg_size_z), si, 'mmi_in'))
    structures.append(box((x2, 0, 0), (mmi_length_um, mmi_width_um, wg_size_z), si, 'mmi_out'))

    # Input and output access waveguides; port centres align to the chosen pitch.
    x_left = x1 - mmi_length_um / 2 - lead_length / 2
    x_right = x2 + mmi_length_um / 2 + lead_length / 2
    for idx, y in enumerate((y1, y2)):
        structures.append(box((x_left, y, 0), (lead_length, wg_width, wg_size_z), si, f'input_lead_{idx}'))
        structures.append(box((x_right, y, 0), (lead_length, wg_width, wg_size_z), si, f'output_lead_{idx}'))

    # Two arms between MMI blocks. Top arm contains the index-perturbed section.
    arm_x0 = x1 + mmi_length_um / 2
    arm_x1 = x2 - mmi_length_um / 2
    arm_span = arm_x1 - arm_x0
    if phase_length > arm_span:
        raise ValueError('phase_length must fit between the MMI regions')
    phase_xc = (arm_x0 + arm_x1) / 2
    passive_span = (arm_span - phase_length) / 2
    for y, tag in ((y1, 'top'), (y2, 'bottom')):
        if passive_span > 0:
            structures.append(box((arm_x0 + passive_span/2, y, 0), (passive_span, wg_width, wg_size_z), si, f'{tag}_left_arm'))
            structures.append(box((arm_x1 - passive_span/2, y, 0), (passive_span, wg_width, wg_size_z), si, f'{tag}_right_arm'))
        arm_medium = td.Medium(permittivity=(n_si + delta_n)**2) if tag == 'top' else si
        structures.append(box((phase_xc, y, 0), (phase_length, wg_width, wg_size_z), arm_medium, f'{tag}_phase_section'))

    x_min = x_left - lead_length/2
    x_max = x_right + lead_length/2
    sim_size = (x_max - x_min + 2*3.0, mmi_width_um + 4.0, 3.0)
    sim_center = ((x_min + x_max)/2, 0, 0)
    source_x = x_left - lead_length/2 + 1.0
    monitor_x = x_right + lead_length/2 - 1.0
    mode_spec = td.ModeSpec(num_modes=2)
    source = td.ModeSource(
        center=(source_x, y1, 0), size=(0, 2.5, 1.5),
        source_time=td.GaussianPulse(freq0=freq0, fwidth=freq0/10),
        direction='+', mode_spec=mode_spec, mode_index=0,
    )
    monitor = td.ModeMonitor(
        center=(monitor_x, 0, 0), size=(0, mmi_width_um + 2.0, 1.5),
        freqs=freqs, mode_spec=mode_spec, name='out_modes',
    )
    return td.Simulation(
        center=sim_center, size=sim_size,
        grid_spec=td.GridSpec.auto(min_steps_per_wvl=grid_steps_per_wvl),
        structures=structures, sources=[source], monitors=[monitor],
        run_time=8e-12,
        boundary_spec=td.BoundarySpec.all_sides(boundary=td.PML()),
        medium=sio2,
    )

sim0 = make_mzi_sim(phase_delta_n)
fig, ax = plt.subplots(figsize=(12, 4))
sim0.plot(z=0, ax=ax)
ax.set_title('RU-002A initial MMI-MZI geometry (not PDK-verified)')
plt.tight_layout()
plt.show()


## 4. Review geometry before dispatch

Inspect the plotted layout and the generated simulation. Confirm that all waveguides connect as intended, the mode source overlaps only the intended input port, the monitor captures the output modes, the mesh resolves the narrowest feature, and the simulation domain has adequate padding. Because this is a simplified rectangle-based MMI model, do not interpret the output as a foundry PDK design.

In [ ]:
print('Simulation size (um):', sim0.size)
print('Structure count:', len(sim0.structures))
print('Source count:', len(sim0.sources), 'Monitor count:', len(sim0.monitors))
print('Remote dispatch remains disabled:', not RUN_REMOTE)


## 5. Optional phase sweep — explicit authorization required

This cell does not run by default. It launches separate FDTD jobs for each phase-index proxy value only after `RUN_REMOTE` is set to `True`. Review cloud cost and confirm the geometry and account credentials first. A production run should use the EIS–FlexAgent authorized adapter and evidence ledger rather than bypassing the conductor.

In [ ]:
phase_delta_n_values = np.linspace(0.0, 0.02, 5)  # numerical proxy sweep only

if RUN_REMOTE:
    results = {}
    for delta_n in phase_delta_n_values:
        sim = make_mzi_sim(float(delta_n))
        task_name = f'RU-002A_MMI_MZI_dn_{delta_n:.4f}'
        results[task_name] = web.run(sim, task_name=task_name, path=str(OUT_DIR / f'{task_name}.hdf5'))
    print('Jobs submitted:', list(results))
else:
    print('No jobs submitted. Set RUN_REMOTE=True only after the EIS execution gate and geometry review pass.')


## 6. Required result checks and evidence ledger

Do not infer a balanced MMI from the parameter values. Once a real run completes, extract complex modal amplitudes at the target wavelength, calculate power per mode, check total transmitted/reflected/absorbed power as supported by the monitor setup, and compare with an independent circuit-level MZI model. Save the actual Tidy3D task IDs, tool version, PDK/material provenance, configuration hash, result hashes, and acceptance verdict. Record failures as failures; no result is not a pass.

### Acceptance gates (to be filled with evidence)
- [ ] Intended PDK/platform and material stack verified on the execution host.
- [ ] MMI splitting ratio measured in simulation and within a predeclared tolerance.
- [ ] Mesh/time convergence checked.
- [ ] Complex output amplitudes and phase retained; power-only plots are insufficient for field-level mapping.
- [ ] Phase sweep matches the independent circuit model within a predeclared tolerance.
- [ ] Simulation run IDs, configs, results, hashes, and provenance stored in the EIS evidence ledger.
- [ ] MLA/KKA comparison remains out of scope until the reduced workload mapping and same-workload electronic baseline are pinned.


## References
- FlexCompute, [Tunable Mach-Zehnder interferometer based on directional couplers](https://www.flexcompute.com/tidy3d/community/notebooks/TunableMZI/).
- FlexCompute, [Tidy3D notebook collection](https://github.com/flexcompute/tidy3d-notebooks).
- EIS–FlexAgent-001 authorized experiment loop: execution must be blocked if the real host lacks verified submission, status, result, credential, or PDK capabilities.
